# Aralin 09 - Padron ng Disenyo ng Metakognisyon


## Pagsasaayos

Ipinapakita ng notebook na ito ang Metacognition design pattern gamit ang Microsoft Agent Framework.

**Mga Kinakailangan:**
- Naka-configure na deployment ng Azure OpenAI sa pamamagitan ng mga environment variable
- Azure CLI na naka-authenticate (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang Metakognisyon?

Ang metakognisyon ay **pag-iisip tungkol sa pag-iisip**. Sa konteksto ng mga ahente ng AI, nangangahulugan ito ng pagbuo ng mga ahenteng maaaring:

- **Magmuni-muni** sa kanilang sariling mga output at proseso ng pangangatwiran
- **Magtukoy ng mga pagkakamali** at makabawi nang maayos sa halip na tahimik na mabigo
- **Suriin** kung ang kanilang mga tugon ay kumpleto at kapaki-pakinabang
- **Umangkop** ang kanilang estratehiya kapag ang paunang pamamaraan ay hindi gumana (hal., pagbalik sa isang backup na sistema)

Ang isang metakognitibong ahente ay hindi lamang sumasagot ng mga tanong — minomonitor nito ang sarili nitong pagganap at agad na umaangkop.


## Pangunahing at Pangalawang Mga Kasangkapan

Isang karaniwang pattern ng metacognition ay ang **estratehiya ng fallback**. Sinusubukan ng ahente ang pangunahing kasangkapan muna; kung ito ay mabibigo (hal., isang 404 error), kinikilala ng ahente ang pagkabigong iyon at hayagang lumilipat sa isang backup na kasangkapan.

Ginagaya nito ang mga totoong sistema kung saan maaaring hindi magamit ang mga pangunahing serbisyo at kailangang sariling matukoy ng mga ahente ang problema bago pumili ng alternatibong landas.

Sa ibaba inilalarawan namin ang dalawang kasangkapan para sa paghahanap ng flight:
- **Pangunahing** — sumasaklaw sa Paris, Tokyo, at Barcelona
- **Pangalawang** — sumasaklaw sa Berlin, Sydney, at New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Ahenteng May Sariling Pagsusuri at Pagbawi ng Error

Ang ahente sa ibaba ay inutusan na subukan muna ang pangunahing sistema ng paglipad, kilalanin ang mga pagkabigo, at hayagang lumipat sa backup na sistema. Pagkatapos ng bawat tugon, maikli nitong sinusuri ang sarili kung nasagot ba nito nang buo ang tanong ng gumagamit.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Modelo ng Pagsusuri sa Sarili

Isa pang aspekto ng metakognisyon ay **pagsusuri sa sarili**: isang hiwalay na ahente (o ang parehong ahente sa pangalawang pagpasa) ang sinusuri ang isang tugon para sa pagiging kumpleto, kawastuhan, at pagiging kapaki-pakinabang.

Sa ibaba gagawa tayo ng isang ahente na `ResponseEvaluator` na nagbibigay ng iskor sa mga tugon ng ahente ng paglalakbay sa tatlong dimensyon.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Buod

Sa leksyong ito natutunan mo kung paano bumuo ng **mga metakognitibong ahente** gamit ang Microsoft Agent Framework:

- **Pagsusuri sa sarili**: Mga ahenteng sumusubaybay sa kanilang sariling pangangatwiran at malinaw na ipinapahayag kung ano ang nangyari.
- **Pagbawi sa error na may fallback**: Isang pattern ng pangunahing + backup na tool kung saan natutukoy ng ahente ang mga pagkabigo (hal., mga error na 404) at awtomatikong sinusubukan ang alternatibong pinagmulan.
- **Pansariling pagsusuri**: Isang hiwalay na ahenteng tagapag-evaluate na nagbibigay ng iskor sa mga sagot para sa pagiging kumpleto, katumpakan, at kapaki-pakinabang.

Pinapahusay ng mga pattern na ito ang pagiging matatag, malinaw, at mapagkakatiwalaan ng mga ahente — mahahalagang katangian para sa mga deployment sa produksyon.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:
Isinalin ang dokumentong ito gamit ang AI na serbisyo ng pagsasalin na Co-op Translator (https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kaming maging tumpak, pakitandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatumpak. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na awtoritatibong sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin na ginagawa ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na nagmumula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
